# Faceless video: image step
Self-contained: no uploads needed. Runtime > Change runtime type > T4 GPU, then Runtime > Run all. The last cell downloads `images.zip`.

In [ ]:
!pip -q uninstall -y torchao  # clashes with diffusers on Colab
!pip -q install -U diffusers transformers accelerate safetensors peft

In [ ]:
%%writefile generate_images.py
"""Scenes JSON -> scene_XX.png with consistent characters. Run on a free Colab/Kaggle GPU (see README).

Characters: one reference image each (SDXL), then every scene uses IP-Adapter conditioned on the
reference of its first listed character plus the character descriptions in the prompt.
"""
import json, sys
from pathlib import Path

import torch
from diffusers import AutoPipelineForText2Image
from PIL import Image

STYLES = {
    "stickman": "simple hand-drawn stickman illustration, thick black lines, flat colors, white background",
    "claymation": "claymation stop-motion style, plasticine figures, soft studio lighting",
    "3d cartoon": "3D cartoon render, pixar-like, soft lighting, vibrant colors",
}


def main(scenes_path, out="images"):
    spec = json.loads(Path(scenes_path).read_text())
    out = Path(out); out.mkdir(exist_ok=True)
    style = STYLES.get(spec.get("style", "stickman"), spec["style"])
    portrait = spec.get("aspect", "9:16") == "9:16"
    size = (768, 1344) if portrait else (1344, 768)
    pipe = AutoPipelineForText2Image.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16").to("cuda")
    pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")
    chars = {c["id"]: c for c in spec["characters"]}
    refs = {}
    pipe.set_ip_adapter_scale(0.0)
    blank = Image.new("RGB", (224, 224), "white")
    for cid, c in chars.items():
        g = torch.Generator("cuda").manual_seed(abs(hash(cid)) % 2**31)
        img = pipe(f"character sheet, one character, full body, centered, {c['description']}, {style}",
                   ip_adapter_image=blank, width=1024, height=1024, generator=g).images[0]
        img.save(out / f"char_{cid}.png"); refs[cid] = img
    for i, sc in enumerate(spec["scenes"]):
        cs = sc.get("characters", [])
        desc = ", ".join(chars[c]["description"] for c in cs)
        pipe.set_ip_adapter_scale(0.6 if cs else 0.0)
        ref = refs[cs[0]] if cs else blank
        g = torch.Generator("cuda").manual_seed(1000 + i)
        img = pipe(f"{sc['image_prompt']}, {desc}, {style}", negative_prompt="text, watermark, blurry, extra limbs",
                   ip_adapter_image=ref, width=size[0], height=size[1], generator=g).images[0]
        img.save(out / f"scene_{i:02d}.png"); print("done", i)


if __name__ == "__main__":
    main(*sys.argv[1:])


In [ ]:
%%writefile scenes.json
{
  "title": "Stickman survival: lost in the forest",
  "aspect": "9:16",
  "style": "stickman",
  "voice": "en_US-lessac-medium",
  "characters": [
    {"id": "sam", "description": "stickman hiker named Sam with a green backpack and an orange beanie"},
    {"id": "owl", "description": "small stickman-style owl with big round eyes and a tiny brown scarf"}
  ],
  "scenes": [
    {"narration": "Sam is lost in the forest, and the sun is going down.", "image_prompt": "lone hiker standing in a dark pine forest at sunset, looking worried", "characters": ["sam"]},
    {"narration": "Rule number one: stop walking. Panic makes you wander farther from help.", "image_prompt": "hiker sitting on a log, hand raised in a stop gesture, calm expression", "characters": ["sam"]},
    {"narration": "Rule two: find water. A stream can lead you toward people.", "image_prompt": "hiker kneeling by a small stream between trees", "characters": ["sam"]},
    {"narration": "A wise old owl watches him from a branch.", "image_prompt": "owl perched on a branch above the hiker, forest at dusk", "characters": ["owl", "sam"]},
    {"narration": "Rule three: build a shelter before dark. Lean branches against a fallen log.", "image_prompt": "hiker leaning branches against a fallen log to make a lean-to shelter", "characters": ["sam"]},
    {"narration": "Rule four: make noise and signals. Three whistles mean help.", "image_prompt": "hiker blowing a whistle, sound waves, owl nearby", "characters": ["sam", "owl"]},
    {"narration": "Stay warm, stay visible, and stay put.", "image_prompt": "hiker sitting by a small campfire under the stars, owl beside him", "characters": ["sam", "owl"]},
    {"narration": "At sunrise, rescuers follow the whistle and find Sam safe.", "image_prompt": "rescuers with flashlights arriving at the shelter at dawn, hiker waving happily", "characters": ["sam"]}
  ]
}


In [ ]:
!python generate_images.py scenes.json images

In [ ]:
from google.colab import files
!zip -qr images.zip images
files.download('images.zip')